# Fink/LSST — Flux Stability Statistics per Gaia Category

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and computes, for each Gaia category, the distribution of the relative flux  
dispersion **σ(F)/⟨F⟩** for the three flux types: `scienceFlux`, `templateFlux`,  
and `psfFlux`.

**Three figures** are produced, one per Gaia category:
- `gaia_star_variable`
- `gaia_nophotgstar_stable_unknown_parallax`
- `gaia_star_stable_hq`

Each figure contains **6 subplots (2 rows × 3 columns)**, one per photometric  
band (u, g, r / i, z, y).  Each subplot shows three overlapping histograms  
of σ(F)/⟨F⟩ (science=blue, template=green, psfFlux=red).

**Outlier rejection** uses percentile clipping (default: 2nd–98th percentile)  
before computing statistics.  A summary table is appended at the end.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/CNRS/IN2P3 — Université Paris-Saclay
- **created** : 2026-05-17
- **subject** : Fink alert Broker applied to Rubin LSST alerts — flux stability statistics

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend (zoom/pan) when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_02c"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Flux-type display settings ─────────────────────────────────────────────────
# Three flux types compared in every subplot
FLUX_TYPES = {
    "scienceFlux": {"col": "r:scienceFlux", "color": "steelblue", "label": "scienceFlux"},
    "templateFlux": {"col": "r:templateFlux", "color": "seagreen", "label": "templateFlux"},
    "psfFlux": {"col": "r:psfFlux", "color": "firebrick", "label": "psfFlux"},
}

# ── Gaia categories of interest ───────────────────────────────────────────────
GAIA_CATEGORIES = [
    "gaia_star_variable",
    "gaia_nophotgstar_stable_unknown_parallax",
    "gaia_star_stable_hq",
]

# ── Outlier rejection parameters ──────────────────────────────────────────────
# Per-object σ(F)/⟨F⟩ values are clipped to the [PCTLO, PCTHI] percentile range
# before computing histogram statistics.  This removes grossly non-physical
# outliers (e.g. objects with near-zero mean flux) without tuning a hard threshold.
PCTLO = 2  # lower percentile
PCTHI = 98  # upper percentile
MIN_PTS = 3  # minimum number of finite positive visits required per object/band

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{ext}")


print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Gaia categories  : {GAIA_CATEGORIES}")

## 2. Utility functions

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def sigma_over_mean(flux_array, min_pts=MIN_PTS):
    """Compute sigma(F)/<F> for a single object light curve (one band, one flux type).

    Parameters
    ----------
    flux_array : array-like
        All visit flux values (nJy) for this object in this band.
    min_pts : int
        Minimum number of valid (finite, positive) data points required.
        Returns NaN if fewer are available.

    Returns
    -------
    float or NaN
        sigma(F)/<F> computed on finite positive flux values.
        NaN if insufficient data.
    """
    f = np.asarray(flux_array, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    if len(f) < min_pts:
        return np.nan
    mean_f = np.mean(f)
    if mean_f <= 0:
        return np.nan
    return float(np.std(f) / mean_f)


def percentile_clip(values, pctlo=PCTLO, pcthi=PCTHI):
    """Return finite values clipped to the [pctlo, pcthi] percentile range.

    Parameters
    ----------
    values : array-like
        Input values (may contain NaN / inf).
    pctlo, pcthi : float
        Lower and upper percentile bounds (0-100).

    Returns
    -------
    ndarray
        Filtered array of finite values within the percentile range.
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if len(v) < 4:
        return v
    lo, hi = np.percentile(v, [pctlo, pcthi])
    return v[(v >= lo) & (v <= hi)]


print("Utility functions defined.")

## 3. Load Parquet files from disk

In [ ]:
# Auto-discover available groups from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))

print(f"Groups found on disk: {all_groups}")

# Only keep the three Gaia categories of interest
available_cats = [g for g in GAIA_CATEGORIES if g in all_groups]
missing_cats = [g for g in GAIA_CATEGORIES if g not in all_groups]
if missing_cats:
    print(f"WARNING: the following categories were NOT found on disk: {missing_cats}")
print(f"Available categories for analysis: {available_cats}")

In [ ]:
# Load Parquet files and reconstruct lc_cache
# Structure: lc_cache[group][oid] = {'fp': DataFrame, 'src': DataFrame}

lc_cache = {}

for group in available_cats:
    lc_cache[group] = {}

    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])

        # Drop rows missing core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Split by diaObjectId
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

# Summary
print("Objects loaded per Gaia category:")
for g in available_cats:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:50s}  {n_oids:4d} objects   {n_pts:7d} data points")

## 4. Compute sigma(F)/<F> per object, per band, per flux type

For each group x band x flux type, we aggregate all per-object sigma(F)/<F> values  
into a single distribution.  The concatenation uses **all visits** for each object  
(both `fp` and `src` DataFrames, deduplicated on timestamp x band).

Outlier objects (near-zero mean flux, <MIN_PTS measurements) are automatically  
excluded by `sigma_over_mean()`.  The resulting distributions are then stored in  
`stat_cache[group][band][flux_type]` as a 1-D array of finite values.

In [ ]:
# stat_cache[group][band][flux_type] = 1-D array of sigma(F)/<F> values (one per object)
# All values are finite (NaN-producing objects are silently dropped).

stat_cache = {}

for group in available_cats:
    stat_cache[group] = {b: {} for b in BANDS}

    for oid, d in lc_cache[group].items():
        # Merge fp + src, deduplicate on (timestamp, band)
        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        df_all = (
            pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
            .dropna(subset=["r:midpointMjdTai"])
            .reset_index(drop=True)
        )

        for band in BANDS:
            df_b = df_all[df_all["r:band"] == band]
            if len(df_b) < MIN_PTS:
                continue

            for ft_name, ft_info in FLUX_TYPES.items():
                col = ft_info["col"]
                if col not in df_b.columns:
                    continue
                som = sigma_over_mean(df_b[col].values)
                if not np.isfinite(som):
                    continue
                if ft_name not in stat_cache[group][band]:
                    stat_cache[group][band][ft_name] = []
                stat_cache[group][band][ft_name].append(som)

    # Convert lists to arrays
    for band in BANDS:
        for ft_name in stat_cache[group][band]:
            stat_cache[group][band][ft_name] = np.array(stat_cache[group][band][ft_name])

# Quick sanity check
print("sigma(F)/<F> sample sizes (before percentile clipping):")
for group in available_cats:
    print(f"  {group}")
    for band in BANDS:
        parts = []
        for ft_name in FLUX_TYPES:
            arr = stat_cache[group][band].get(ft_name, np.array([]))
            parts.append(f"{ft_name}:{len(arr)}")
        print(f"    band {band}: {' | '.join(parts)}")

## 5. Histograms of sigma(F)/<F> — one figure per Gaia category

Layout: **2 rows x 3 columns** (u, g, r / i, z, y).  
Colours: science=steelblue, template=seagreen, psfFlux=firebrick.  
The legend in each subplot shows the mean of the (clipped) distribution.  
Outlier rejection: percentile clipping at [PCTLO, PCTHI]%.


In [ ]:
def plot_sigma_histograms(group, n_bins=40, log_x=False):
    """Plot 2x3 histograms of sigma(F)/<F> for a given Gaia category.

    Parameters
    ----------
    group : str
        Gaia category key (must be in stat_cache).
    n_bins : int
        Number of histogram bins.
    log_x : bool
        If True, use a logarithmic x-axis (useful for variable stars).
    """
    if group not in stat_cache:
        print(f"Group {group!r} not in stat_cache — skipping.")
        return

    band_layout = [
        ("u", 0, 0),
        ("g", 0, 1),
        ("r", 0, 2),
        ("i", 1, 0),
        ("z", 1, 1),
        ("y", 1, 2),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8), gridspec_kw={"hspace": 0.45, "wspace": 0.35})

    for band, row, col in band_layout:
        ax = axes[row, col]
        color_band = BAND_COLORS.get(band, "gray")

        for ft_name, ft_info in FLUX_TYPES.items():
            raw = stat_cache[group][band].get(ft_name, np.array([]))
            if len(raw) < 2:
                continue

            # Percentile clipping
            clipped = percentile_clip(raw)
            if len(clipped) < 2:
                continue

            mean_val = np.mean(clipped)
            std_val = np.std(clipped)
            n_obj = len(clipped)

            label = f"{ft_info['label']}\n  <s/F>={mean_val:.4f}\n  rms={std_val:.4f} N={n_obj}"

            if log_x:
                # log-spaced bins for variable stars (wide dynamic range)
                pos = clipped[clipped > 0]
                if len(pos) < 2:
                    continue
                bins = np.logspace(np.log10(pos.min()), np.log10(pos.max()), n_bins)
                ax.hist(
                    pos, bins=bins, color=ft_info["color"], alpha=0.45, histtype="stepfilled", label=label
                )
                ax.hist(pos, bins=bins, color=ft_info["color"], alpha=0.9, histtype="step", linewidth=1.2)
                ax.set_xscale("log")
            else:
                ax.hist(
                    clipped,
                    bins=n_bins,
                    color=ft_info["color"],
                    alpha=0.45,
                    histtype="stepfilled",
                    label=label,
                )
                ax.hist(
                    clipped, bins=n_bins, color=ft_info["color"], alpha=0.9, histtype="step", linewidth=1.2
                )

            # Vertical dashed line at the mean value
            ax.axvline(mean_val, color=ft_info["color"], lw=1.5, ls="--", alpha=0.85)

        # Axes decoration
        ax.set_title(f"band  {band}", color=color_band, fontsize=11, fontweight="bold")
        ax.set_xlabel(r"$\sigma(F) / \langle F \rangle$", fontsize=10)
        ax.set_ylabel("N objects", fontsize=10)
        ax.tick_params(labelsize=9)

        # Legend inside the subplot (mean and rms displayed there)
        ax.legend(
            fontsize=7.5,
            loc="upper right",
            framealpha=0.85,
            handlelength=1.2,
            borderpad=0.6,
            labelspacing=0.35,
        )

    # Figure-level title: the Gaia category name
    fig.suptitle(
        f"Relative flux dispersion  $\\sigma(F)/\\langle F \\rangle$\nGaia category: {group}",
        fontsize=13,
        fontweight="bold",
        y=1.01,
    )

    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02c_sigma_hist_{safe}")
    plt.show()


print("plot_sigma_histograms() defined.")

### 5a. Variable stars — `gaia_star_variable`

In [ ]:
if "gaia_star_variable" in available_cats:
    plot_sigma_histograms("gaia_star_variable", n_bins=20, log_x=False)
else:
    print("gaia_star_variable not available in this dataset.")

### 5b. Stable stars with unknown parallax — `gaia_nophotgstar_stable_unknown_parallax`

In [ ]:
if "gaia_nophotgstar_stable_unknown_parallax" in available_cats:
    plot_sigma_histograms("gaia_nophotgstar_stable_unknown_parallax", n_bins=20, log_x=False)
else:
    print("gaia_nophotgstar_stable_unknown_parallax not available in this dataset.")

### 5c. High-quality stable stars — `gaia_star_stable_hq`

In [ ]:
if "gaia_star_stable_hq" in available_cats:
    plot_sigma_histograms("gaia_star_stable_hq", n_bins=20, log_x=False)
else:
    print("gaia_star_stable_hq not available in this dataset.")

## 6. Summary table

Statistics of σ(F)/⟨F⟩ after percentile clipping, organised with the following
three-level hierarchy:

| Level | Key | Order |
|-------|-----|-------|
| **Top** (flux type) | `flux_type` | psfFlux → scienceFlux → templateFlux |
| **Middle** (band) | `band` | u → g → r → i → z → y |
| **Inner** (Gaia category, rows close together) | `gaia_category` | gaia_star_stable_hq → gaia_nophotgstar_stable_unknown_parallax → gaia_star_variable |

Columns reported per row:
- **N_obj** : objects entering the computation (≥ MIN_PTS visits, before clipping)
- **N_clip** : objects retained after percentile clipping [PCTLO%, PCTHI%]
- **mean σ/F** : mean of the clipped distribution
- **rms σ/F** : standard deviation of the clipped distribution
- **median σ/F** : median of the clipped distribution

All values are dimensionless ratios.

In [ ]:
# ── Ordered categories for the three index levels ────────────────────────────
# Level 1 (top grouping): flux type
FLUX_ORDER = ["psfFlux", "scienceFlux", "templateFlux"]

# Level 2 (middle): photometric band
BAND_ORDER = list("ugrizy")

# Level 3 (inner, rows in close proximity): Gaia category
GAIA_ORDER = [
    "gaia_star_stable_hq",
    "gaia_nophotgstar_stable_unknown_parallax",
    "gaia_star_variable",
]

# Short labels for display (full names are very wide)
SHORT_LABEL = {
    "gaia_star_stable_hq": "stable_hq",
    "gaia_nophotgstar_stable_unknown_parallax": "stable_unkn_par",
    "gaia_star_variable": "variable",
}

# ── Collect statistics — outer loop = flux_type, then band, then gaia_category
rows = []

for ft_name in FLUX_ORDER:
    for band in BAND_ORDER:
        for group in GAIA_ORDER:
            if group not in stat_cache:
                continue
            raw = stat_cache[group][band].get(ft_name, np.array([]))
            n_obj = len(raw)
            if n_obj == 0:
                continue
            clipped = percentile_clip(raw)
            n_clip = len(clipped)
            if n_clip < 2:
                continue
            rows.append(
                {
                    "flux_type": ft_name,
                    "band": band,
                    "gaia_category": group,
                    "N_obj": n_obj,
                    "N_clip": n_clip,
                    "mean_sigma_F": float(np.mean(clipped)),
                    "rms_sigma_F": float(np.std(clipped)),
                    "median_sigma_F": float(np.median(clipped)),
                }
            )

df_summary = pd.DataFrame(rows)

# Enforce the desired sort order via Categorical dtypes
df_summary["flux_type"] = pd.Categorical(df_summary["flux_type"], categories=FLUX_ORDER, ordered=True)
df_summary["band"] = pd.Categorical(df_summary["band"], categories=BAND_ORDER, ordered=True)
df_summary["gaia_category"] = pd.Categorical(df_summary["gaia_category"], categories=GAIA_ORDER, ordered=True)

df_summary = df_summary.sort_values(["flux_type", "band", "gaia_category"]).reset_index(drop=True)

print(
    f"Summary table: {len(df_summary)} rows  "
    f"({len(FLUX_ORDER)} flux types x {len(BAND_ORDER)} bands x {len(GAIA_ORDER)} categories)"
)
print(df_summary[["flux_type", "band", "gaia_category"]].drop_duplicates().head(18).to_string())

### 6b. Full styled table — MultiIndex (flux_type, band, gaia_category)

The table is re-indexed on `(flux_type, band, gaia_category)` so the three-level
hierarchy is immediately visible.  A colour gradient is applied to `mean σ/F`
**independently within each flux-type block** (green = small dispersion,
red = large dispersion), making within-block comparisons meaningful.

In [ ]:
# Build display copy with short Gaia labels
df_disp = df_summary.copy()
df_disp["gaia_category"] = (
    df_disp["gaia_category"].map(SHORT_LABEL).fillna(df_disp["gaia_category"].astype(str))
)
df_disp["gaia_category"] = pd.Categorical(
    df_disp["gaia_category"],
    categories=[SHORT_LABEL[g] for g in GAIA_ORDER if g in SHORT_LABEL],
    ordered=True,
)
df_disp = df_disp.set_index(["flux_type", "band", "gaia_category"])


def style_table(df):
    """Return a styled DataFrame with per-flux_type gradient on mean_sigma_F."""
    styled = df.style.format({"mean_sigma_F": "{:.5f}", "rms_sigma_F": "{:.5f}", "median_sigma_F": "{:.5f}"})
    # Apply gradient row-slice per flux_type independently so that colours
    # are comparable within each flux-type block, not across all blocks.
    for ft in FLUX_ORDER:
        lvl0 = df.index.get_level_values("flux_type")
        if ft not in lvl0:
            continue
        row_pos = [i for i, v in enumerate(lvl0) if v == ft]
        styled = styled.background_gradient(
            subset=pd.IndexSlice[df.index[row_pos], "mean_sigma_F"],
            cmap="RdYlGn_r",
            axis=0,
        )
    styled = styled.set_caption(
        f"σ(F)/⟨F⟩ statistics  |  percentile clip [{PCTLO}%, {PCTHI}%]  |  MIN_PTS={MIN_PTS}\n"
        "Hierarchy: flux_type (top)  >  band (middle)  >  gaia_category (inner)"
    )
    return styled


pd.options.display.max_rows = 300
display(style_table(df_disp))

### 6c. Compact pivot — mean σ/F only

Rows: `(flux_type, gaia_category)` in the requested hierarchy order.  
Columns: bands u → g → r → i → z → y.  
This removes N_obj / rms / median to give the densest possible overview,
ideal for a quick comparison across flux types and bands.

In [ ]:
df_piv = df_summary.copy()
df_piv["gaia_category"] = df_piv["gaia_category"].map(SHORT_LABEL).fillna(df_piv["gaia_category"].astype(str))

pivot = df_piv.pivot_table(
    index=["flux_type", "gaia_category"],
    columns="band",
    values="mean_sigma_F",
    aggfunc="mean",
).reindex(columns=BAND_ORDER)

# Enforce row order: flux_type (top) then gaia_category (inner)
row_order = [
    (ft, SHORT_LABEL[g]) for ft in FLUX_ORDER for g in GAIA_ORDER if (ft, SHORT_LABEL[g]) in pivot.index
]
pivot = pivot.reindex(row_order)

print("Mean σ(F)/⟨F⟩  |  rows: (flux_type, gaia_category)  |  columns: band (u→g→r→i→z→y)")
display(
    pivot.style.format("{:.5f}")
    .background_gradient(cmap="YlOrRd", axis=None)
    .set_caption(
        "Compact pivot — mean σ(F)/⟨F⟩\n"
        "Row hierarchy: flux_type (top) > gaia_category (inner) | columns: bands"
    )
)

### 6d. Export summary table to CSV

In [ ]:
csv_out = os.path.join(DIR_FIGS, "02c_sigma_summary.csv")
# Save with string categories (not Categorical dtype) for portability
df_summary.astype({"flux_type": str, "band": str, "gaia_category": str}).to_csv(csv_out, index=False)
print(f"Summary table saved to: {csv_out}")
df_summary.head(12)